# 15 A stronger API
The API had two inputs, `product_id` and `num_similar`. Here I make it stronger: clear input checks and error codes, a `mode` option (exact or fast), a `weights` option, and a small endpoint that gives the page what it needs. Every request below is a real request to the real service (the final search, with the vectors that ship with it), made through FastAPI's test client. Nothing here uses the 20 locked final products.

In [1]:
import os
from pathlib import Path

import pandas as pd
from fastapi.testclient import TestClient

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(root)  # the project reads data/archive.zip relative to the project folder
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 110)

from product_similarity.api import create_app  # noqa: E402
from product_similarity.service import build_final_engine  # noqa: E402

engine = build_final_engine("exact")
engine.prepare_fast()  # in Docker this happens when the service starts
PID = "26d41bdc1495de290bc8e6062d927729"
client = TestClient(create_app(lambda: engine, cache_size=0))
client.__enter__()
print("the service is ready:", client.get("/ready").json())

the service is ready: {'status': 'ready', 'products': 29529, 'search': 'hybrid (exact)', 'modes': ['exact', 'fast'], 'fast_ready': True}


## 1. The parameters
| Parameter | Meaning | Rule |
|---|---|---|
| `product_id` | The product to start from (required) | 32 lowercase hexadecimal characters |
| `num_similar` | How many similar products to return (required) | A whole number from 1 to 1,000 |
| `mode` | `exact` or `fast` (optional) | The default is `exact` |
| `weights` | How much each part counts (optional) | `name:value` pairs, each value from 0 to 1. If it is left out, the tuned weights are used |

**Weights, two ways.** The *simple* way has 4 names: `title`, `text_vectors`, `image` and `attributes` (the seven attributes together). The *advanced* way has the 10 parts: `title`, `text_vectors`, `image`, `category`, `brand`, `audience`, `price`, `colour`, `weight`, `rating`. The two ways cannot be mixed. What you send is what is used, and a part you leave out gets 0. The weights do not have to add up to 1, because the score is a weighted average and only the proportions matter. At least one weight must be above 0. The page gets the names, the range and the tuned weights from `GET /weights`.

In [2]:
info = client.get("/weights").json()
print(
    "modes:",
    info["modes"],
    "| default mode:",
    info["default_mode"],
    "| most results:",
    info["max_results"],
)
print("tuned weights, the simple way:  ", info["tuned_weights"]["simple"])
print("tuned weights, the advanced way:", info["tuned_weights"]["advanced"])

modes: ['exact', 'fast'] | default mode: exact | most results: 1000
tuned weights, the simple way:   {'title': 0.028571, 'text_vectors': 0.457143, 'image': 0.285714, 'attributes': 0.228571}
tuned weights, the advanced way: {'category': 0.07033, 'brand': 0.035165, 'audience': 0.035165, 'price': 0.035165, 'colour': 0.017582, 'weight': 0.017582, 'rating': 0.017582, 'title': 0.028571, 'text_vectors': 0.457143, 'image': 0.285714}


## 2. Errors
Every error has the same shape: a short `error` code and a plain sentence in `message`. Here is each kind, asked for real:

In [3]:
base = "/find_similar_products"
cases = [
    ("num_similar is 0", {"product_id": PID, "num_similar": 0}),
    ("num_similar is too large", {"product_id": PID, "num_similar": 5000}),
    ("num_similar is not a number", {"product_id": PID, "num_similar": "many"}),
    ("product_id is not 32 hexadecimal characters", {"product_id": "abc", "num_similar": 5}),
    ("product_id is missing", {"num_similar": 5}),
    ("a product that does not exist", {"product_id": "0" * 32, "num_similar": 5}),
    ("an unknown mode", {"product_id": PID, "num_similar": 5, "mode": "turbo"}),
    ("a weight above 1", {"product_id": PID, "num_similar": 5, "weights": "image:2"}),
    ("an unknown weight", {"product_id": PID, "num_similar": 5, "weights": "speed:1"}),
    ("all weights are 0", {"product_id": PID, "num_similar": 5, "weights": "image:0,title:0"}),
    (
        "simple and advanced mixed",
        {"product_id": PID, "num_similar": 5, "weights": "attributes:0.5,brand:0.5"},
    ),
]
rows = []
for what, params in cases:
    r = client.get(base, params=params)
    body = r.json()
    rows.append(
        {
            "request": what,
            "status": r.status_code,
            "error": body["error"],
            "message": body["message"],
        }
    )
display(pd.DataFrame(rows).set_index("request"))

,status,error,message
request,,,
num_similar is 0,422,invalid_input,num_similar must be a whole number from 1 to 1000
num_similar is too large,422,invalid_input,num_similar must be a whole number from 1 to 1000
num_similar is not a number,422,invalid_input,num_similar must be a whole number from 1 to 1000
product_id is not 32 hexadecimal characters,422,invalid_input,product_id must be 32 lowercase hexadecimal characters
product_id is missing,422,invalid_input,product_id is required
a product that does not exist,404,product_not_found,no product has the ID 00000000000000000000000000000000
an unknown mode,422,invalid_mode,"mode must be one of: exact, fast"
a weight above 1,422,invalid_weights,"the weight of 'image' must be from 0 to 1, not 2"
an unknown weight,422,invalid_weights,"unknown weight 'speed'; the simple names are title, text_vectors, image, attributes and the advanced names..."


### What this shows
A bad input gets a 422 with a sentence that says what to change, an unknown product gets a 404, and a service that has not loaded yet gets a 503 (the tests cover that one). An unexpected error gives a 500 with a safe message, and the details go to the log only. Unknown addresses and wrong methods have the same JSON shape.

## 3. The `mode` option
The exact mode compares the sentence and picture vectors with every product. The fast mode uses the two HNSW indexes for those two parts (notebook 12). The answer says which mode ran, in the header `X-Search-Mode` and in the `mode` field of the details endpoint. The cache is switched off here, so that every request is a real search and the time is the time of the search.

In [4]:
def details(**extra):
    return client.get(
        base + "/details", params={"product_id": PID, "num_similar": 3, **extra}
    ).json()


rows = []
for mode in ("exact", "fast"):
    answer = details(mode=mode)
    rows.append(
        {
            "mode asked": mode,
            "mode that ran": answer["mode"],
            "time (ms)": answer["response_time_ms"],
            "first three results": " | ".join(r["title"][:34] for r in answer["results"]),
        }
    )
display(pd.DataFrame(rows).set_index("mode asked"))

,mode that ran,time (ms),first three results
mode asked,,,
exact,exact,12.93,Kalamkari Print Cotton Fabric for | Cotton Kalamkari Handblock Saree B | Worldoftextile Women's Cotton Sang
fast,fast,1.91,Cotton Kalamkari Handblock Saree B | Worldoftextile Women's Cotton Sang | INAYA HAND BLOCK Printed Cotton Sa


### What this shows
The fast mode ran faster, and it returned the same kind of products. For this one product, the first result of the two modes differs: the fast mode makes its shortlist from the sentence and picture vectors only, so it can miss a product that the exact mode ranks high mainly through its attributes. Notebooks 12 and 14 measure the two modes with my ratings, and the quality is the same for both. The time of a single request is noisy, so the measured times are in those notebooks.

## 4. The `weights` option
The same product, searched with different weights. The exact mode is used, and each request carries its own weights, so nothing is saved between requests.

In [5]:
settings = [
    ("no weights (the tuned weights)", {}),
    ("picture only: image:1", {"weights": "image:1"}),
    ("title only: title:1", {"weights": "title:1"}),
    ("brand only (advanced): brand:1", {"weights": "brand:1"}),
    ("sentence 0.5 and picture 0.5", {"weights": "text_vectors:0.5,image:0.5"}),
]
print("query:", details()["query"]["title"])
rows = []
for label, extra in settings:
    answer = details(**extra)
    rows.append(
        {
            "weights": label,
            "the three most similar products": " | ".join(
                r["title"][:42] for r in answer["results"]
            ),
        }
    )
display(pd.DataFrame(rows).set_index("weights"))

query: LA' Facon Cotton Kalamkari Handblock Saree Blouse Fabric 100 cms Black Base Dancers (Cotton)


,the three most similar products
weights,
no weights (the tuned weights),Kalamkari Print Cotton Fabric for Women's | Cotton Kalamkari Handblock Saree Blouse/Ku | Worldoftextile W...
picture only: image:1,Riyasat - Self Pattern Blue Colour Micro F | Riyasat - Floral Pattern Pink Color Micro | Alvaro Castagnin...
title only: title:1,Cotton Kalamkari Handblock Saree Blouse/Ku | HANDBLOCK PRINT Women's Rayon Embroidered | Winza Designer W...
brand only (advanced): brand:1,Cotton Kalamkari Handblock Saree Blouse/Ku | HANCOCK Yellow Solid Slim Fit Cotton Rich | Indistar Girl's ...
sentence 0.5 and picture 0.5,Kalamkari Print Cotton Fabric for Women's | Worldoftextile Women's Cotton Sanganeri Ha | Ramdev Crafts Wo...


### What this shows
Different weights give different lists, as they should. With the picture only, the products look like the query but can be a different kind of product. With the title only, they are the same kind of fabric. With the brand only, only the brand counts, so the products have little else in common. If both vector weights are 0 and `mode=fast` is asked, the fast mode cannot make its shortlist, so the exact mode runs and the answer says so:

In [6]:
answer = details(mode="fast", weights="brand:1")
print("mode asked: fast | mode that ran:", answer["mode"])
print("note:", answer["mode_note"])

mode asked: fast | mode that ran: exact
note: the fast mode needs a sentence or picture weight above 0, so the exact mode ran


## 5. The cache question
The cache remembers an answer for a question. The question is now the product, the number of results, the mode and the weights. The same question gives a hit, and another mode or other weights is another question, so an answer for one set of weights is never given for another. Weights written in another order are the same question.

In [7]:
cached = TestClient(create_app(lambda: engine, cache_size=1024))
cached.__enter__()


def ask(label, **extra):
    r = cached.get(base, params={"product_id": PID, "num_similar": 3, **extra})
    return {
        "request": label,
        "X-Cache": r.headers["X-Cache"],
        "mode that ran": r.headers["X-Search-Mode"],
    }


rows = [
    ask("tuned weights, exact"),
    ask("tuned weights, exact (again)"),
    ask("picture only", weights="image:1"),
    ask("picture only (again)", weights="image:1"),
    ask("title only", weights="title:1"),
    ask("tuned weights, fast", mode="fast"),
    ask("sentence and picture", weights="text_vectors:0.5,image:0.5"),
    ask("same, written in another order", weights="image:0.5,text_vectors:0.5"),
]
display(pd.DataFrame(rows).set_index("request"))
print(cached.get("/cache").json())

,X-Cache,mode that ran
request,,
"tuned weights, exact",miss,exact
"tuned weights, exact (again)",hit,exact
picture only,miss,exact
picture only (again),hit,exact
title only,miss,exact
"tuned weights, fast",miss,fast
sentence and picture,miss,exact
"same, written in another order",hit,exact


{'enabled': True, 'max_entries': 1024, 'entries': 5, 'hits': 3, 'misses': 5, 'hit_rate': 0.375}


### What this shows
An answer is a hit only for exactly the same question. The first two rows are the same question (a miss and then a hit), and the other weights and the other mode each start as a miss. The last two rows are the same weights in another order, so the second is a hit.

## 6. Conclusion

**What I did.** I made the API stronger.

- Bad input now gets a clear message and the right code: 422 for a bad input, 404 for an unknown product, 503 when the service is not ready. `num_similar` must be from 1 to 1,000.
- You can now choose the `mode` (exact or fast) for each search.
- You can now set your own `weights` for each search: the simple way with 4 names, or the advanced way with 10, each from 0 to 1.
- The cache remembers a different answer for each mode and each set of weights.
- A new `GET /weights` gives the tuned weights for the page, and in Docker the fast mode is ready as soon as the service starts.

**What I do next.** Kubernetes and the CI/CD improvements, then the page, and the final cleanup.